# 02 – Mô hình baseline (Logistic Regression)

**Baseline** là mô hình đơn giản đầu tiên, dùng làm *mốc để so sánh*. Các thuật toán khác ở notebook 03 phải thắng được baseline thì mới có ý nghĩa.

**Quy trình học máy cổ điển cho ảnh:**
1. **Trích xuất đặc trưng** (feature engineering): biến mỗi ảnh thành một vector 50 số (histogram màu HSV, mean/std màu, tỉ lệ vùng tối/nâu, texture). Xem `src/features.py`.
2. **Chuẩn hoá** đặc trưng (`StandardScaler`): đưa về mean 0, std 1.
3. **Huấn luyện** bộ phân loại trên vector đặc trưng. Baseline ở đây là **Logistic Regression**.
4. **Đánh giá** bằng accuracy, precision, recall, F1, confusion matrix.

**Lưu ý:** notebook này **không ghi đè** `models/best_model.joblib`. Biểu đồ và báo cáo được lưu vào `reports/` với tiền tố `baseline_`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # để import được package src

import numpy as np
import pandas as pd
from IPython.display import Image

from src.data_loader import REPORTS_DIR
from src.evaluate import evaluate_model
from src.features import FEATURE_NAMES, load_feature_splits
from src.train import DEFAULT_CONFIG, fit_model

cfg = DEFAULT_CONFIG
print(cfg)

## 1. Trích xuất đặc trưng và chia dữ liệu

`load_feature_splits()` chia dữ liệu thành 3 tập với 3 vai trò khác nhau:

| Tập | Vai trò |
|---|---|
| **train** | Để thuật toán học |
| **val** | Để so sánh các thuật toán / chỉnh siêu tham số |
| **test** | Chỉ dùng cuối cùng, một lần, để báo cáo kết quả |

Lần chạy đầu sẽ mất vài phút để trích xuất đặc trưng, các lần sau đọc lại từ `data/processed/features_*.npz`.

In [ ]:
data = load_feature_splits()
X_train, y_train = data['train']
X_val, y_val = data['val']
X_test, y_test = data['test']
print('X_train:', X_train.shape, '| X_val:', X_val.shape, '| X_test:', X_test.shape)

# So sánh đặc trưng trung bình của hai nhãn (0 = fresh, 1 = rotten) để xem đặc trưng nào phân biệt tốt
df = pd.DataFrame(X_train, columns=FEATURE_NAMES)
df['label'] = y_train
diff = (df.groupby('label').mean().T.assign(chenh_lech=lambda d: (d[1] - d[0]).abs())
        .sort_values('chenh_lech', ascending=False))
diff.head(10)

## 2. Huấn luyện baseline

**Logistic Regression** học một đường (siêu phẳng) tách hai lớp trong không gian 50 chiều, và cho ra xác suất thuộc mỗi lớp bằng hàm sigmoid. Tham số `C` điều khiển mức regularization (C nhỏ = ràng buộc mạnh hơn, mô hình đơn giản hơn).

In [ ]:
model, val = fit_model(cfg, X_train, y_train, X_val, y_val)
print('Validation:', val)

## 3. Đánh giá trên tập test

- **Accuracy:** tỉ lệ dự đoán đúng.
- **Precision:** trong số ảnh model gọi là `rotten`, bao nhiêu % đúng là hỏng.
- **Recall:** trong số ảnh thật sự hỏng, model bắt được bao nhiêu %.
- **Confusion matrix:** đếm số ảnh đúng/sai của từng lớp.

In [ ]:
test = evaluate_model(model, X_test, y_test, prefix='baseline_')
Image(os.path.join(REPORTS_DIR, 'baseline_confusion_matrix.png'))

## 4. Hệ số của Logistic Regression

Vì đặc trưng đã được chuẩn hoá nên có thể so sánh độ lớn hệ số: hệ số dương lớn nghĩa là đặc trưng đó làm tăng khả năng ảnh là `rotten`, hệ số âm lớn thì ngược lại (`fresh`).

In [ ]:
coef = pd.Series(model.named_steps['clf'].coef_[0], index=FEATURE_NAMES).sort_values()
print('Thiên về FRESH nhất:\n', coef.head(5), '\n')
print('Thiên về ROTTEN nhất:\n', coef.tail(5))